In [1]:
import numpy as np

In [73]:

#definição de matriz x 3x2

x = np.array([[1, 2], [3, 4], [5, 6]])
c = np.array([[1, 2, 3], [4, 5, 6]])

y = np.array([[1], [2]])
b = np.array([1, 2])

print(y.shape)

(2, 1)


In [79]:
xy = np.einsum('ij,ij->j', x, y)

ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (3,2)->(2,3) (2,1)->(1,2) 

In [80]:
xy

array([ 5, 11, 17])

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. Definição da Arquitetura da Rede Neural
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

# Configurando sementes para reprodutibilidade pura
torch.manual_seed(42)
np.random.seed(42)

# 2. Dados de Treinamento (Apenas 8 pontos)
x_train_np = np.linspace(-np.pi, np.pi, 8).reshape(-1, 1)
y_train_np = np.sin(x_train_np)
dy_train_np = np.cos(x_train_np)

x_train = torch.FloatTensor(x_train_np).requires_grad_(True)
y_train = torch.FloatTensor(y_train_np)
dy_train = torch.FloatTensor(dy_train_np)

# Instanciando os modelos
model_normal = SimpleNet()
model_sobolev = SimpleNet()

optimizer_normal = optim.Adam(model_normal.parameters(), lr=0.01)
optimizer_sobolev = optim.Adam(model_sobolev.parameters(), lr=0.01)
criterion = nn.MSELoss()

print(f"{'Época':<7} | {'Loss Normal':<15} | {'Sobolev (Valor)':<15} | {'Sobolev (Derivada)':<18}")
print("-" * 65)

# 3. Loop de Treinamento
epochs = 1001
for epoch in range(epochs):
    # --- TREINO NORMAL ---
    optimizer_normal.zero_grad()
    pred_normal = model_normal(x_train)
    loss_normal = criterion(pred_normal, y_train)
    loss_normal.backward()
    optimizer_normal.step()
    
    # --- TREINO SOBOLEV ---
    optimizer_sobolev.zero_grad()
    
    # Criamos um clone para evitar interferência de gradientes cruzados
    x_sobolev = x_train.clone().detach().requires_grad_(True)
    pred_sobolev = model_sobolev(x_sobolev)
    
    # Coleta o primeiro elemento da tupla retornada pelo autograd
    dy_pred = torch.autograd.grad(
        outputs=pred_sobolev, 
        inputs=x_sobolev,
        grad_outputs=torch.ones_like(pred_sobolev),
        create_graph=True,
        retain_graph=True
    )[0]
    
    loss_val = criterion(pred_sobolev, y_train)
    loss_deriv = criterion(dy_pred, dy_train)
    loss_total_sobolev = loss_val + 0.5 * loss_deriv
    
    loss_total_sobolev.backward()
    optimizer_sobolev.step()
    
    # --- PRINTS DETALHADOS ---
    if epoch % 100 == 0:
        print(f"{epoch:<7} | {loss_normal.item():<15.6f} | {loss_val.item():<15.6f} | {loss_deriv.item():<18.6f}")

print("-" * 65)
print("Treinamento concluído!")


Época   | Loss Normal     | Sobolev (Valor) | Sobolev (Derivada)
-----------------------------------------------------------------
0       | 0.887141        | 0.555489        | 0.565219          
100     | 0.005723        | 0.000019        | 0.000051          
200     | 0.000116        | 0.000006        | 0.000001          
300     | 0.000012        | 0.000003        | 0.000001          
400     | 0.000002        | 0.000807        | 0.000018          
500     | 0.000000        | 0.000001        | 0.000001          
600     | 0.000000        | 0.000010        | 0.000001          
700     | 0.000001        | 0.000001        | 0.000001          
800     | 0.000000        | 0.000095        | 0.000006          
900     | 0.000000        | 0.000001        | 0.000001          
1000    | 0.000005        | 0.000011        | 0.000003          
-----------------------------------------------------------------
Treinamento concluído!
